In [ ]:
# =========================================================
# Dataset S1 – LMM trend estimation for Richness
# =========================================================

library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(marginaleffects)
library(MuMIn)

# =========================================================

df <- read_csv("D:/NC/Data/rivernet/inputdata/Richness.csv")

df <- df %>%
  filter(year >= 1982, year <= 2018) %>%
  mutate(
    period = case_when(
      year <= 2004 ~ "early",
      year >= 2005 ~ "late",
      TRUE ~ NA_character_
    ),
    period = factor(period, levels = c("early", "late")),
    zone   = factor(zone, levels = c("Cold", "Intermediate", "Warm")),
    year_c = year - mean(year, na.rm = TRUE),
    SiteID = factor(SiteID),
    Protocol = factor(Protocol),
    HYBAS_ID = factor(HYBAS_ID)
  ) %>%
  filter(
    !is.na(period),
    !is.na(zone),
    !is.na(species_richness)
  )

df_all   <- df
df_early <- df %>% filter(period == "early")
df_late  <- df %>% filter(period == "late")

data_list <- list(
  early = df_early,
  late  = df_late,
  full  = df_all
)

# =========================================================
# model fitting functions
# =========================================================

fit_all <- function(dat) {
  lmer(
    species_richness ~ year_c +
      (1 + year_c || SiteID) +
      (1 | HYBAS_ID) +
      (1 | Protocol),
    data = dat,
    REML = FALSE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )
}

fit_zone <- function(dat) {
  lmer(
    species_richness ~ year_c * zone +
      (1 + year_c || SiteID) +
      (1 | HYBAS_ID) +
      (1 | Protocol),
    data = dat,
    REML = FALSE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )
}

# =========================================================

models_all  <- lapply(data_list, fit_all)
models_zone <- lapply(data_list, fit_zone)

# =========================================================

slopes_all <- lapply(
  models_all,
  avg_slopes,
  variables = "year_c"
)

slopes_zone <- lapply(
  models_zone,
  avg_slopes,
  variables = "year_c",
  by = "zone"
)

tab_all <- bind_rows(
  early = slopes_all$early,
  late  = slopes_all$late,
  full  = slopes_all$full,
  .id = "period"
) %>%
  transmute(
    period,
    zone = "All",
    slope = estimate,
    SE = std.error,
    z = statistic,
    p = p.value
  )

tab_zone <- bind_rows(
  early = slopes_zone$early,
  late  = slopes_zone$late,
  full  = slopes_zone$full,
  .id = "period"
) %>%
  transmute(
    period,
    zone,
    slope = estimate,
    SE = std.error,
    z = statistic,
    p = p.value
  )

tab_slopes_all <- bind_rows(tab_all, tab_zone) %>%
  arrange(period, zone) %>%
  mutate(
    sig = case_when(
      p < 0.001 ~ "***",
      p < 0.01  ~ "**",
      p < 0.05  ~ "*",
      TRUE ~ ""
    )
  )

write_csv(
  tab_slopes_all,
  "D:/outputdata/Trend estimation using LMMs/Richness_slopes_summary.csv"
)

# =========================================================

out_path <- "D:/outputdata/Trend estimation using LMMs/"

save_model_output <- function(model, name) {

  capture.output(
    {
      cat("========================================\n")
      cat("Model:", name, "\n")
      cat("========================================\n\n")

      print(summary(model))

      cat("\n----------------------------------------\n")
      cat("Model fit statistics\n")
      cat("----------------------------------------\n")

      cat("AIC:", AIC(model), "\n")
      cat("BIC:", BIC(model), "\n")

      r2 <- r.squaredGLMM(model)
      cat("R2_marginal:", r2[1], "\n")
      cat("R2_conditional:", r2[2], "\n")
    },
    file = paste0(out_path, name, ".txt")
  )
}

# overall
save_model_output(models_all$early, "Richness_LMM_overall_early")
save_model_output(models_all$late,  "Richness_LMM_overall_late")
save_model_output(models_all$full,  "Richness_LMM_overall_full")

# zone
save_model_output(models_zone$early, "Richness_LMM_zone_early")
save_model_output(models_zone$late,  "Richness_LMM_zone_late")
save_model_output(models_zone$full,  "Richness_LMM_zone_full")

Rows: 43034 Columns: 25
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (7): SiteID, Protocol, FlowClass, elev_class, HFP_class, geometry, zone
dbl (18): year, annual_temp, Longitude, Latitude, species_richness, seg_id_n...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
"Model failed to converge with max|grad| = 0.00214226 (tol = 0.002, component 1)"
Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
"Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?"
Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
"Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?"
Warning message:
"For this model type, `marginaleffects` only takes i

In [ ]:
# =========================================================
# Dataset S1 – LMM trend estimation for Abundance
# =========================================================

library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(marginaleffects)
library(MuMIn)

# =========================================================

df <- read_csv("D:/NC/Data/rivernet/inputdata/Abundance.csv")

df <- df %>%
  filter(year >= 1982, year <= 2018) %>%
  mutate(
    period = case_when(
      year <= 2004 ~ "early",
      year >= 2005 ~ "late",
      TRUE ~ NA_character_
    ),
    period = factor(period, levels = c("early", "late")),
    zone   = factor(zone, levels = c("Cold", "Intermediate", "Warm")),
    year_c = year - mean(year, na.rm = TRUE),
    SiteID = factor(SiteID),
    Species = factor(Species),
    Protocol = factor(Protocol),
    HYBAS_ID = factor(HYBAS_ID)
  ) %>%
  filter(
    !is.na(period),
    !is.na(zone),
    !is.na(abundance_std)
  )

df_all   <- df
df_early <- df %>% filter(period == "early")
df_late  <- df %>% filter(period == "late")

data_list <- list(
  early = df_early,
  late  = df_late,
  full  = df_all
)

# =========================================================

fit_all <- function(dat) {
  lmer(
    abundance_std ~ year_c +
      (1 + year_c || SiteID) +
      (1 | Species) +
      (1 | Protocol) +
      (1 | HYBAS_ID),
    data = dat,
    REML = FALSE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )
}

fit_zone <- function(dat) {
  lmer(
    abundance_std ~ year_c * zone +
      (1 + year_c || SiteID) +
      (1 | Species) +
      (1 | Protocol) +
      (1 | HYBAS_ID),
    data = dat,
    REML = FALSE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )
}

# =========================================================

models_all  <- lapply(data_list, fit_all)
models_zone <- lapply(data_list, fit_zone)

# =========================================================

slopes_all <- lapply(
  models_all,
  avg_slopes,
  variables = "year_c"
)

slopes_zone <- lapply(
  models_zone,
  avg_slopes,
  variables = "year_c",
  by = "zone"
)

tab_all <- bind_rows(
  early = slopes_all$early,
  late  = slopes_all$late,
  full  = slopes_all$full,
  .id = "period"
) %>%
  transmute(
    period,
    zone = "All",
    slope = estimate,
    SE = std.error,
    z = statistic,
    p = p.value
  )

tab_zone <- bind_rows(
  early = slopes_zone$early,
  late  = slopes_zone$late,
  full  = slopes_zone$full,
  .id = "period"
) %>%
  transmute(
    period,
    zone,
    slope = estimate,
    SE = std.error,
    z = statistic,
    p = p.value
  )

tab_slopes_all <- bind_rows(tab_all, tab_zone) %>%
  arrange(period, zone) %>%
  mutate(
    sig = case_when(
      p < 0.001 ~ "***",
      p < 0.01  ~ "**",
      p < 0.05  ~ "*",
      TRUE ~ ""
    )
  )

write_csv(
  tab_slopes_all,
  "D:/outputdata/Trend estimation using LMMs/Abundance_slopes_summary.csv"
)

# =========================================================

out_path <- "D:/outputdata/Trend estimation using LMMs/"

save_model_output <- function(model, name) {

  capture.output(
    {
      cat("========================================\n")
      cat("Model:", name, "\n")
      cat("========================================\n\n")

      print(summary(model))

      cat("\n----------------------------------------\n")
      cat("Model fit statistics\n")
      cat("----------------------------------------\n")

      cat("AIC:", AIC(model), "\n")
      cat("BIC:", BIC(model), "\n")

      r2 <- r.squaredGLMM(model)
      cat("R2_marginal:", r2[1], "\n")
      cat("R2_conditional:", r2[2], "\n")
    },
    file = paste0(out_path, name, ".txt")
  )
}

# overall
save_model_output(models_all$early, "Abundance_LMM_overall_early")
save_model_output(models_all$late,  "Abundance_LMM_overall_late")
save_model_output(models_all$full,  "Abundance_LMM_overall_full")

# zone
save_model_output(models_zone$early, "Abundance_LMM_zone_early")
save_model_output(models_zone$late,  "Abundance_LMM_zone_late")
save_model_output(models_zone$full,  "Abundance_LMM_zone_full")

Warning message:
"One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)"
Rows: 245462 Columns: 25
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (10): SiteID, Species, Protocol, UnitAbundance, TimeSeriesID, FlowClass,...
dbl (15): year, Abundance, annual_temp, Longitude, Latitude, Richness, seg_i...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
boundary (singular) fit: see help('isSingular')

Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
"Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?"
boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
"Model is nearly unidentif

In [ ]:
# =========================================================
# Dataset – LMM trend estimation for LCBD (1982–2018)
# =========================================================

library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(marginaleffects)
library(MuMIn)

# =========================================================

df <- read_csv("D:/NC/Data/rivernet/inputdata/Richness.csv")

df_lcbd <- df %>%
  filter(year >= 1982, year <= 2018) %>%
  mutate(
    zone   = factor(zone, levels = c("Cold", "Intermediate", "Warm")),
    year_c = year - mean(year, na.rm = TRUE),
    SiteID = factor(SiteID),
    Protocol = factor(Protocol),
    HYBAS_ID = factor(HYBAS_ID)
  ) %>%
  filter(!is.na(LCBD), !is.na(zone))

# =========================================================

m_lcbd_zone <- lmer(
  LCBD ~ year_c * zone +
    (1 + year_c || SiteID) +
    (1 | HYBAS_ID) +
    (1 | Protocol),
  data = df_lcbd,
  REML = FALSE,
  control = lmerControl(
    optimizer = "bobyqa",
    optCtrl = list(maxfun = 2e5)
  )
)

m_lcbd_all <- lmer(
  LCBD ~ year_c +
    (1 + year_c || SiteID) +
    (1 | HYBAS_ID) +
    (1 | Protocol),
  data = df_lcbd,
  REML = FALSE,
  control = lmerControl(
    optimizer = "bobyqa",
    optCtrl = list(maxfun = 2e5)
  )
)

# =========================================================

slp_zone <- avg_slopes(
  m_lcbd_zone,
  variables = "year_c",
  by = "zone"
)

tab_zone <- slp_zone %>%
  transmute(
    period = "full",
    zone,
    slope = estimate,
    SE = std.error,
    z = statistic,
    p = p.value
  )

slp_all <- avg_slopes(
  m_lcbd_all,
  variables = "year_c"
)

tab_all <- slp_all %>%
  transmute(
    period = "full",
    zone = "All",
    slope = estimate,
    SE = std.error,
    z = statistic,
    p = p.value
  )

tab_lcbd <- bind_rows(tab_all, tab_zone) %>%
  arrange(zone) %>%
  mutate(
    sig = case_when(
      p < 0.001 ~ "***",
      p < 0.01  ~ "**",
      p < 0.05  ~ "*",
      TRUE ~ ""
    )
  )

# =========================================================

out_path <- "D:/outputdata/Trend estimation using LMMs/"

write_csv(
  tab_lcbd,
  paste0(out_path, "LCBD_LMM_slopes_full.csv")
)

# =========================================================

save_model_output <- function(model, name) {

  capture.output(
    {
      cat("========================================\n")
      cat("Model:", name, "\n")
      cat("========================================\n\n")

      print(summary(model))

      cat("\n----------------------------------------\n")
      cat("Model fit statistics\n")
      cat("----------------------------------------\n")

      cat("AIC:", AIC(model), "\n")
      cat("BIC:", BIC(model), "\n")

      r2 <- r.squaredGLMM(model)
      cat("R2_marginal:", r2[1], "\n")
      cat("R2_conditional:", r2[2], "\n")
    },
    file = paste0(out_path, name, ".txt")
  )
}

save_model_output(m_lcbd_all,  "LCBD_LMM_overall_full")
save_model_output(m_lcbd_zone, "LCBD_LMM_zone_full")


Rows: 43034 Columns: 25
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (7): SiteID, Protocol, FlowClass, elev_class, HFP_class, geometry, zone
dbl (18): year, annual_temp, Longitude, Latitude, species_richness, seg_id_n...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to

In [ ]:
# =========================================================
# Dataset – LMM trend estimation for Functional Diversity (SES_FDis)
# =========================================================

library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(marginaleffects)
library(MuMIn)

# =========================================================

df <- read_csv("D:/NC/Data/rivernet/inputdata/Richness.csv")

df_fd <- df %>%
  filter(year >= 1982, year <= 2018) %>%
  mutate(
    zone   = factor(zone, levels = c("Cold", "Intermediate", "Warm")),
    year_c = year - mean(year, na.rm = TRUE),
    SiteID = factor(SiteID),
    Protocol = factor(Protocol),
    HYBAS_ID = factor(HYBAS_ID)
  ) %>%
  filter(!is.na(SES_FDis), !is.na(zone))

# =========================================================

m_fd_zone <- lmer(
  SES_FDis ~ year_c * zone +
    (1 + year_c || SiteID) +
    (1 | HYBAS_ID) +
    (1 | Protocol),
  data = df_fd,
  REML = FALSE,
  control = lmerControl(
    optimizer = "bobyqa",
    optCtrl = list(maxfun = 2e5)
  )
)

m_fd_all <- lmer(
  SES_FDis ~ year_c +
    (1 + year_c || SiteID) +
    (1 | HYBAS_ID) +
    (1 | Protocol),
  data = df_fd,
  REML = FALSE,
  control = lmerControl(
    optimizer = "bobyqa",
    optCtrl = list(maxfun = 2e5)
  )
)

# =========================================================

slp_zone <- avg_slopes(
  m_fd_zone,
  variables = "year_c",
  by = "zone"
)

tab_zone <- slp_zone %>%
  transmute(
    period = "full",
    zone,
    slope = estimate,
    SE = std.error,
    z = statistic,
    p = p.value
  )

slp_all <- avg_slopes(
  m_fd_all,
  variables = "year_c"
)

tab_all <- slp_all %>%
  transmute(
    period = "full",
    zone = "All",
    slope = estimate,
    SE = std.error,
    z = statistic,
    p = p.value
  )

tab_fd <- bind_rows(tab_all, tab_zone) %>%
  arrange(zone) %>%
  mutate(
    sig = case_when(
      p < 0.001 ~ "***",
      p < 0.01  ~ "**",
      p < 0.05  ~ "*",
      TRUE ~ ""
    )
  )

# =========================================================

out_path <- "D:/outputdata/Trend estimation using LMMs/"

write_csv(
  tab_fd,
  paste0(out_path, "FD_LMM_slopes_full.csv")
)

# =========================================================

save_model_output <- function(model, name) {

  capture.output(
    {
      cat("========================================\n")
      cat("Model:", name, "\n")
      cat("========================================\n\n")

      print(summary(model))

      cat("\n----------------------------------------\n")
      cat("Model fit statistics\n")
      cat("----------------------------------------\n")

      cat("AIC:", AIC(model), "\n")
      cat("BIC:", BIC(model), "\n")

      r2 <- r.squaredGLMM(model)
      cat("R2_marginal:", r2[1], "\n")
      cat("R2_conditional:", r2[2], "\n")
    },
    file = paste0(out_path, name, ".txt")
  )
}

save_model_output(m_fd_all,  "FD_LMM_overall_full")
save_model_output(m_fd_zone, "FD_LMM_zone_full")

Rows: 43034 Columns: 25
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (7): SiteID, Protocol, FlowClass, elev_class, HFP_class, geometry, zone
dbl (18): year, annual_temp, Longitude, Latitude, species_richness, seg_id_n...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
"Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?"
Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
"Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?"
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or t